# V7_A_N07 — Market Prices With Context

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft using synthetic data. Outputs support authorized review only.

## Decision contract
Monitor commodity prices, verify abnormal movements, and inform market preparedness. Owners: market-information, trade, agriculture, and statistical authorities. A price signal does not authorize trade restrictions, procurement, or enforcement.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(7507); dates=pd.date_range('2025-01-01',periods=72,freq='W'); markets=['Central','North','South']; rows=[]
for m in markets:
 for i,d in enumerate(dates): rows.append((d,m,'Maize','kg','retail',210+.7*i+12*np.sin(2*np.pi*i/26)+rng.normal(0,7)))
df=pd.DataFrame(rows,columns=['date','market','commodity','unit','level','price_lcu']); df.loc[(df.market=='North')&(df.date>=dates[-5]),'price_lcu']+=80; df.tail()

## Evidence contract and comparability
Commodity, variety/grade, unit, transaction level, market, currency, collection method, and reference date must match. Nominal changes should be separated from general inflation where relevant.

In [2]:
assert df[['commodity','unit','level']].nunique().eq(1).all(); cpi=pd.Series(np.linspace(100,108,len(dates)),index=dates);df['cpi']=df.date.map(cpi);df['real_price']=df.price_lcu/(df.cpi/100);print(df[['price_lcu','real_price']].describe().round(1))

       price_lcu  real_price
count      216.0       216.0
mean       237.4       228.0
std         21.8        17.3
min        200.4       199.3
25%        222.6       216.6
50%        234.3       225.8
75%        249.6       235.7
max        339.7       314.5


## Transparent seasonal baseline
Use only earlier observations in each market. A robust rolling median reduces sensitivity to isolated reporting errors.

In [3]:
df=df.sort_values(['market','date']);df['baseline']=df.groupby('market').real_price.transform(lambda s:s.shift(1).rolling(13,min_periods=8).median());df['mad']=df.groupby('market').real_price.transform(lambda s:(s-s.shift(1).rolling(13,min_periods=8).median()).abs().shift(1).rolling(13,min_periods=8).median());df['robust_z']=(df.real_price-df.baseline)/(1.4826*df.mad.clip(lower=1));df.tail(10)[['date','market','real_price','baseline','robust_z']].round(2)

## Verification signals and missing-market control
An anomaly can reflect real scarcity, unit/grade change, collection error, market closure, transport disruption, or policy change. Route it to verification with reason codes.

In [4]:
df['signal']=df.robust_z.abs()>=3; alerts=df[df.signal].copy();alerts['status']='VERIFY PRICE, UNIT, GRADE, MARKET CONDITIONS';alerts['authority']='Market information unit';print(alerts.tail(10)[['date','market','real_price','robust_z','status']].round(2).to_string(index=False))

      date market  real_price  robust_z                                       status
2026-04-19  North      299.08      4.85 VERIFY PRICE, UNIT, GRADE, MARKET CONDITIONS
2026-04-26  North      302.48      5.13 VERIFY PRICE, UNIT, GRADE, MARKET CONDITIONS
2026-05-17  North      314.51      3.29 VERIFY PRICE, UNIT, GRADE, MARKET CONDITIONS


## Distribution and policy boundary
A national average can conceal local and household consequences. Report dispersion and avoid deriving import/export quantities or enforcement actions from one series.

In [5]:
latest=df[df.date==df.date.max()];summary={'median_real_price':round(latest.real_price.median(),1),'market_range':round(latest.real_price.max()-latest.real_price.min(),1),'alerts':int(latest.signal.sum()),'prohibited_use':'automatic trade restriction, procurement, or market enforcement'};print(summary)

{'median_real_price': np.float64(233.5), 'market_range': np.float64(85.8), 'alerts': 1, 'prohibited_use': 'automatic trade restriction, procurement, or market enforcement'}


## Sensitivity
Compare 2.5 and 3.5 robust-z thresholds. A policy workflow must balance verification burden against missed disruptions.

In [6]:
sens=pd.DataFrame({'threshold':[2.5,3,3.5],'alerts':[(df.robust_z.abs()>=x).sum() for x in [2.5,3,3.5]]});print(sens.to_string(index=False))

 threshold  alerts
       2.5       7
       3.0       3
       3.5       2


## Exercises
1. Add wholesale and farmgate series without mixing levels. 2. Harmonize a 100-kg sack to kilograms. 3. Compare nominal and real alerts. 4. Explain why price weights cannot become budget shares automatically.

## Exact solutions
1. Preserve transaction level and compare within compatible panels. 2. Divide the sack price by 100 only after validating net weight and grade. 3. Deflate with a documented index and report both interpretations. 4. Prices do not encode objectives, needs, distribution, externalities, constraints, uncertainty, or legal authority.

In [7]:
assert len(alerts)>0 and 'automatic' in summary['prohibited_use'];print('V7_A_N07_REWORK_COMPLETE_EXECUTION_PASS')

V7_A_N07_REWORK_COMPLETE_EXECUTION_PASS
